# Feature Engineering

Turn raw bills into clean model inputs. Output: `peakflo_features.parquet`.

In [8]:
import html
import json
import re

import numpy as np
import pandas as pd

df = pd.json_normalize(json.load(open("accounts-bills.json")))
df["itemDescription"] = df["itemDescription"].fillna("")
df.shape

(4894, 7)

## 1. Clean text

Names start with a bill month like `0825`, and contain invoice numbers. We keep only words so the model learns *what* was bought.
Description is added only when it says something new (83% are copies of the name).

In [9]:
ABBR = {"fb": "facebook", "ig": "instagram", "cdn": "content delivery network",
        "sms": "short message service", "atm": "cash", "sem": "search engine marketing"}


def clean(s):
    s = html.unescape(s.lower())                # "&amp;" -> "&"
    s = re.sub(r"^\s*\d{4}\b[\s\-_:]*", "", s)  # drop "0825 " month prefix
    words = re.findall(r"[a-z]+", s)            # letters only, numbers gone
    return " ".join(ABBR.get(w, w) for w in words)


def make_text(name, desc):
    name, desc = clean(name), clean(desc)
    if desc and not (name.startswith(desc) or desc.startswith(name)):
        return f"{name} {desc}".strip()
    return max(name, desc, key=len) or "nodescription"


df["text"] = [make_text(n, d) for n, d in zip(df["itemName"], df["itemDescription"])]
df[["itemName", "text"]].head()

,itemName,text
0,1225 Business 2Gbps - monthly subscription fee,business gbps monthly subscription fee
1,1225-0227 media monitoring,media monitoring
2,0226 Branch,branch
3,0126 GAM (SG),gam sg
4,0126 GAM (Chotot & Mudah),gam chotot mudah


## 2. Bill month

The `MMYY` prefix tells us when the bill is from. We use it to put future months in the test set.

In [10]:
m = df["itemName"].str.extract(r"^\s*(\d{2})(\d{2})\b")  # "0825" -> "08", "25"
ok = pd.to_numeric(m[0], errors="coerce").between(1, 12) & m[1].isin(["23", "24", "25", "26"])
df["ym"] = np.where(ok, "20" + m[1] + "-" + m[0], None)  # "2025-08"

print(f"bills with a month: {df['ym'].notna().mean():.1%}")

bills with a month: 87.3%


## 3. Service period

Prepaid bills often cover a range like `0126-1226`; online subscriptions almost never do.
`has_range` and `span` (length in months, log) capture that.

In [11]:
r = df["itemName"].str.extract(r"^\s*(\d{2})(\d{2})\s*[-–]\s*(\d{2})(\d{2})\b").apply(pd.to_numeric, errors="coerce")
months = (r[3] * 12 + r[2]) - (r[1] * 12 + r[0]) + 1

df["has_range"] = months.notna().astype(int)
df["span"] = np.log1p(months.clip(lower=0).fillna(0))

df.groupby("accountName")["has_range"].mean().sort_values(ascending=False).head()

accountName
134004 Prepaid Subscription         0.761905
619208 Insurance                    0.666667
134001 Prepaid Operating Expense    0.554404
134002 Prepaid Insurance            0.333333
619502 Bank Charges                 0.277778
Name: has_range, dtype: float64

## 4. Amount

Amounts go from -15k to 161M. Log scale + a 1M cap stops a few huge bills from dominating. `is_neg` flags refunds.

In [12]:
amount = pd.to_numeric(df["itemTotalAmount"], errors="coerce").fillna(0)
capped = amount.clip(-1_000_000, 1_000_000)

df["log_amount"] = np.sign(capped) * np.log10(1 + capped.abs())
df["is_neg"] = (amount < 0).astype(int)

df[["itemTotalAmount", "log_amount", "is_neg"]].describe()

,itemTotalAmount,log_amount,is_neg
count,4.894000e+03,4894.000000,4894.000000
mean,9.266691e+04,2.870129,0.003882
std,3.037295e+06,0.979194,0.062193
min,-1.519500e+04,-4.181729,0.000000
25%,1.549700e+02,2.193041,0.000000
50%,8.452000e+02,2.927473,0.000000
75%,3.507140e+03,3.545077,0.000000
max,1.618380e+08,6.000000,1.000000


## 5. Save

In [13]:
cols = ["vendorId", "itemName", "itemTotalAmount", "accountId", "accountName",
        "text", "ym", "has_range", "span", "log_amount", "is_neg"]
df[cols].to_parquet("peakflo_features.parquet", index=False)
df[cols].shape

(4894, 11)